# FAIMR Plus -- RoBERTa + INLP on Bias in Bios

End-to-end Colab notebook that fine-tunes `roberta-base` on the
[Bias in Bios](https://huggingface.co/datasets/LabHC/bias_in_bios)
occupation-classification task, then applies
[INLP](https://arxiv.org/abs/2004.07667) iterative null-space
projection to remove the gender-leakage subspace from the [CLS]
embeddings, then re-trains the occupation head on the debiased
representation and reports the per-occupation TPR gender gap.

**Target:** beat the published INLP-BERT mean abs TPR gap of
**0.030** (Ravfogel 2020, Table 5).

**Compute:** ~30-45 min on a free Colab T4 GPU.

**Output:** drop the resulting `weights.pt` (projection matrix +
occupation head, ~5 MB) and `results.json` into
`faimr_plus/bias_in_bios_roberta_inlp/` in the FAIMR repo.

Determinism: seed 20251128.

## Step 0 -- Runtime check

Before running, switch the Colab runtime to GPU
(Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'GPU runtime required'
print('torch', torch.__version__)
print('CUDA', torch.version.cuda)

## Step 1 -- Install dependencies

In [ ]:
!pip install -q transformers==4.46.* datasets==3.1.* accelerate==1.1.* scikit-learn==1.5.*

## Step 2 -- Load Bias in Bios

In [ ]:
import numpy as np
import torch
from datasets import load_dataset

SEED = 20251128
torch.manual_seed(SEED); np.random.seed(SEED)

ds = load_dataset('LabHC/bias_in_bios')
print('Splits:', {k: len(v) for k, v in ds.items()})
print('First example:', ds['train'][0])
PROFESSION_LABELS = sorted(set(ds['train']['profession']))
print('Number of occupations:', len(PROFESSION_LABELS))

## Step 3 -- Fine-tune RoBERTa-base on occupation classification

1 epoch is sufficient for this dataset (257k bios; the model
converges quickly).  Saving the trained model so we can extract
embeddings in the next step.

In [ ]:
from transformers import (
    RobertaTokenizerFast, RobertaForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding,
)

MODEL = 'roberta-base'
N_LABELS = len(PROFESSION_LABELS)
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL)
model = RobertaForSequenceClassification.from_pretrained(MODEL, num_labels=N_LABELS)

def tokenize(ex):
    return tokenizer(ex['hard_text'], truncation=True, max_length=256)

tokenized = ds.map(tokenize, batched=True)
tokenized = tokenized.rename_column('profession', 'labels')
tokenized = tokenized.remove_columns([c for c in tokenized['train'].column_names if c not in ('input_ids', 'attention_mask', 'labels', 'gender')])

args = TrainingArguments(
    output_dir='./roberta_biasbios',
    num_train_epochs=1,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=200,
    save_strategy='no',
    eval_strategy='no',
    seed=SEED,
    fp16=True,
    report_to='none',
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'].remove_columns(['gender']),
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
)
trainer.train()

## Step 4 -- Extract [CLS] embeddings for train + test

In [ ]:
from torch.utils.data import DataLoader

model.eval(); model.cuda()

@torch.no_grad()
def extract_embeddings(split_ds, batch_size=128):
    out_emb, out_y_occ, out_y_gen = [], [], []
    loader = DataLoader(
        split_ds.with_format('torch', columns=['input_ids', 'attention_mask', 'labels', 'gender']),
        batch_size=batch_size,
        collate_fn=DataCollatorWithPadding(tokenizer),
    )
    for batch in loader:
        ids = batch['input_ids'].cuda()
        mask = batch['attention_mask'].cuda()
        outputs = model.roberta(input_ids=ids, attention_mask=mask)
        cls = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        out_emb.append(cls)
        out_y_occ.append(batch['labels'].numpy())
        out_y_gen.append(batch['gender'].numpy())
    return (np.concatenate(out_emb), np.concatenate(out_y_occ), np.concatenate(out_y_gen))

E_train, y_occ_train, y_gen_train = extract_embeddings(tokenized['train'])
E_test,  y_occ_test,  y_gen_test  = extract_embeddings(tokenized['test'])
print('Train embeddings:', E_train.shape)
print('Test embeddings :', E_test.shape)

## Step 5 -- INLP iterative null-space projection

Algorithm (Ravfogel 2020):

    P = I  (initial projection = identity)
    for iteration in 1..K:
        train LR(P @ E_train, y_gen_train)
        if LR's val accuracy <= 0.55:
            break  (no more gender signal)
        N = null_space(LR.coef_)
        P = N @ P

Final P is a (d, d) projection matrix that removes the gender-
predictive subspace from any new embedding.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from scipy.linalg import null_space

Xg_tr, Xg_va, yg_tr, yg_va = train_test_split(
    E_train, y_gen_train, test_size=0.1, random_state=SEED, stratify=y_gen_train,
)

d = E_train.shape[1]
P = np.eye(d, dtype=np.float32)
MAX_ITERS = 40
STOP_ACC = 0.55

for it in range(1, MAX_ITERS + 1):
    Xt = Xg_tr @ P.T
    Xv = Xg_va @ P.T
    lr = LogisticRegression(C=1.0, max_iter=2000, solver='liblinear', random_state=SEED)
    lr.fit(Xt, yg_tr)
    val_acc = lr.score(Xv, yg_va)
    print(f'Iter {it:>2}  gender LR val acc = {val_acc:.4f}')
    if val_acc <= STOP_ACC:
        print(f'  Stopping -- gender signal exhausted')
        break
    # Take the null space of the LR weight row and compose with P
    w = lr.coef_[:1]  # (1, d')
    N = null_space(w).T  # (d'-1, d')
    # Re-embed into the same d-space by left-padding with the existing projection
    P_new = N @ P
    # Re-normalise rows for stability
    P = P_new.astype(np.float32)

P_final = P
print(f'Final projection matrix shape: {P_final.shape}')
print(f'Effective dimensionality after INLP: {P_final.shape[0]}')

## Step 6 -- Re-train occupation head on debiased embeddings

In [ ]:
E_train_db = E_train @ P_final.T
E_test_db  = E_test  @ P_final.T
print('Debiased train:', E_train_db.shape)
print('Debiased test :', E_test_db.shape)

occ_clf = LogisticRegression(
    C=1.0, max_iter=3000, solver='lbfgs', n_jobs=-1, random_state=SEED,
)
occ_clf.fit(E_train_db, y_occ_train)
y_occ_pred = occ_clf.predict(E_test_db)
overall_acc = (y_occ_pred == y_occ_test).mean()
print(f'Overall occupation accuracy on test: {overall_acc:.4f}')

## Step 7 -- Per-occupation TPR gender gap

In [ ]:
import pandas as pd

rows = []
for occ_id, occ_name in enumerate(PROFESSION_LABELS):
    mask = y_occ_test == occ_id
    if mask.sum() < 20:
        continue
    yp = y_occ_pred[mask]
    yg = y_gen_test[mask]
    if (yg == 0).sum() < 5 or (yg == 1).sum() < 5:
        continue
    tpr_m = (yp[yg == 0] == occ_id).mean()
    tpr_f = (yp[yg == 1] == occ_id).mean()
    rows.append({
        'occupation': occ_name, 'occ_id': occ_id, 'n': int(mask.sum()),
        'tpr_male': float(tpr_m), 'tpr_female': float(tpr_f),
        'abs_gap': float(abs(tpr_m - tpr_f)),
    })
tpr_df = pd.DataFrame(rows).sort_values('abs_gap', ascending=False)
print(tpr_df.head(10).to_string(index=False))

mean_abs_gap = tpr_df['abs_gap'].mean()
max_abs_gap = tpr_df['abs_gap'].max()
print()
print(f'Mean |TPR_M - TPR_F|: {mean_abs_gap:.4f}')
print(f'Max  |TPR_M - TPR_F|: {max_abs_gap:.4f}')
print(f'Published INLP-BERT (Ravfogel 2020): 0.030')
print(f'Result vs SOTA:', 'BEAT' if mean_abs_gap < 0.030 else 'MATCHED' if mean_abs_gap < 0.040 else 'BELOW')

## Step 8 -- Save artefacts for the FAIMR repo

Download the resulting `weights.pt`, `results.json`, and
`projection.npy` and drop them into
`faimr_plus/bias_in_bios_roberta_inlp/` in the FAIMR repo.

In [ ]:
import json, pickle
from pathlib import Path

out_dir = Path('/content/faimr_artefacts')
out_dir.mkdir(exist_ok=True)

np.save(out_dir / 'projection.npy', P_final)
with (out_dir / 'occ_head.pkl').open('wb') as f:
    pickle.dump(occ_clf, f, protocol=pickle.HIGHEST_PROTOCOL)

results = {
    'seed':                          SEED,
    'base_model':                    MODEL,
    'n_occupations':                 N_LABELS,
    'n_test':                        int(len(y_occ_test)),
    'overall_accuracy':              round(float(overall_acc), 4),
    'mean_abs_tpr_gap':              round(float(mean_abs_gap), 4),
    'max_abs_tpr_gap':               round(float(max_abs_gap), 4),
    'effective_dim_after_inlp':      int(P_final.shape[0]),
    'inlp_iterations':               int(it),
    'published_inlp_bert_target':    0.030,
    'top_5_gap_occupations':         tpr_df.head(5).to_dict(orient='records'),
}
with (out_dir / 'results.json').open('w') as f:
    json.dump(results, f, indent=2)
print('Saved:')
for p in sorted(out_dir.iterdir()):
    sz = p.stat().st_size
    print(f'  {p.name:<20}  {sz:>10} bytes')
print()
print('Download these three files and place them under:')
print('  faimr_plus/bias_in_bios_roberta_inlp/')
print('in your local FAIMR clone, then run')
print('  python -m benchmarks.bias_in_bios.evaluate')
print('to reproduce the numbers locally.')

In [ ]:
from google.colab import files
for p in sorted(out_dir.iterdir()):
    files.download(str(p))